# 01 Watersheds
**Series:** Pine Ridge Hydrology  
**Author:** Lilly Jones, PhD                          
**Primary Focus:** Pine Ridge Reservation/Oglala Sioux Tribe  
**Collective:** Oglala Lakota  
**Data Sources:** Census TIGER AIANNH, USGS NHD, USGS WBD

## Purpose
Before analyzing water data, we need to understand the spatial relationships
that shape water on the Pine Ridge study area:
- How do HUC watershed boundaries relate to Tribal territorial boundaries?
- Which stream networks flow through or adjacent to the Pine Ridge study area?
- Where is the monitoring infrastructure, and where are the gaps?

## Framing
Watersheds are defined by topography as water flows based on the shape
of the land. Tribal territories are defined by treaty, history, and
relationship to place. These two systems of organization rarely align.

A watershed may encompass portions of multiple Tribal Nations, multiple
states, and large areas of non-Tribal land. A single Tribal Nation may
span multiple watersheds. This notebook makes those relationships visible
so that subsequent analysis can account for them.

## Research Questions
- Which HUC-8 watersheds overlap Pine Ridge?
- Which streams cross the Pine Ridge Reservation?
- Where does USGS monitoring exist, and where are the gaps?
- What does the monitoring gap look like as an equity map?

## Learning Objectives

By the end of this notebook, learners will be able to:

- distinguish a Census statistical boundary from legal jurisdiction and community-defined lands
- inspect watershed overlap, stream networks, and public monitoring coverage at multiple spatial scales
- document how an off-boundary feature is relevant without representing it as an OST feature

## Prerequisites and Timing

Allow approximately 75–100 minutes. Before beginning, activate the repository environment, read the series governance statement, and complete the preceding notebook where applicable. Work in pairs and rotate analyst, data-steward, skeptic, and documentarian roles.

## Governance Checkpoint

This notebook uses public environmental data describing Oglala Lakota lands and waters. Public availability does not establish permission for every reuse or interpretation. Do not add OST-controlled data, sensitive locations, or community knowledge. Results are educational and screening-level pending OLC/OST review.

In [ ]:
# Imports
import sys
from pathlib import Path
import requests
from io import BytesIO

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
from datetime import datetime

import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

from src.constants import (
    CRS_GEOGRAPHIC, CRS_PROJECTED,
    STUDY_AREA_CENSUS_NAMES, CENSUS_TO_COMMON, OUTPUTS_DIR, FIGURES_DIR,
)
from src.config import load_config, streamflow_site_ids, streamflow_site_names

CONFIG = load_config()
STUDY_BBOX = tuple(CONFIG["study_area"]["hydrologic_context_bbox"])
STUDY_NAMES = [CONFIG["study_area"]["people"]]
STUDY_CENTROIDS = {CONFIG["study_area"]["people"]: CONFIG["study_area"]["centroid"]}
PINE_RIDGE_STREAMGAGES = {
    site["name"]: str(site["id"]) for site in CONFIG["usgs_streamflow_sites"]
}

from src.loaders import (
    load_tribal_boundaries,
    load_nhd_flowlines,
    load_huc_boundary,
    load_usgs_groundwater_sites,
)
from src.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# print(f"Repo root : {REPO_ROOT}")
print(f"Analysis  : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("Imports complete.")

In [ ]:
# Print data sovereignty statement at the top of every notebook
print_data_acknowledgment(
    source_keys=["census_aiannh", "nhd_flowlines", "wbd_huc",
                 "usgs_nwis_groundwater", "usgs_nwis_streamflow"]
)

## Load Oglala Lakota Boundaries

In [ ]:
# Load the Census TIGER AIANNH boundary for the Pine Ridge study area
# First run: downloads ~30MB ZIP, caches locally. Subsequent runs: instant.

study_boundary = load_tribal_boundaries()

print(f"Pine Ridge study boundary loaded: {len(study_boundary)}")
print()
print(study_boundary[["NAME", "common_name", "area_km2"]].to_string(index=False))
print()
print("Note: Census boundaries are for statistical purposes only.")
print("They do not represent legal jurisdiction or Tribal self-definition.")

In [ ]:
# Total land base in context
total_km2 = study_boundary["area_km2"].sum()
primary   = study_boundary[study_boundary["common_name"].isin(STUDY_NAMES)]

print("PINE RIDGE LAND BASE (Census boundaries)")
print(f"  Census statistical boundary area: {total_km2:,.0f} km²")
print(f"  ({total_km2 / 2.59:,.0f} square miles)")
print()
for _, row in primary.iterrows():
    print(f"  {row['common_name']}: {row['area_km2']:,.0f} km² "
          f"({row['area_km2']/2.59:,.0f} sq mi)")

In [ ]:
# Check what layers exist on the WBD service
r = requests.get(
    "https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer",
    params={"f": "json"},
    timeout=30,
)
layers = r.json().get("layers", [])
for l in layers:
    print(f"  id={l['id']}  name={l['name']}")

## Load Watershed (HUC-8) Boundaries

In [ ]:
# Load HUC-8 watersheds for the Pine Ridge hydrologic context
# Per-Nation bounding boxes to avoid API timeouts on large areas

huc8_parts = []
for name, centroid in STUDY_CENTROIDS.items():
    lat, lon = centroid["lat"], centroid["lon"]
    bbox = (lon - 1.5, lat - 1.0, lon + 1.5, lat + 1.0)
    try:
        huc = load_huc_boundary(bbox=bbox, huc_level=8)
        if not huc.empty:
            huc["queried_for"] = name
            huc8_parts.append(huc)
        print(f"  {name}: {len(huc)} HUC-8 units")
    except Exception as e:
        print(f"  {name}: failed — {e}")

if huc8_parts:
    huc8 = pd.concat(huc8_parts, ignore_index=True)
    # Deduplicate on HUC code
    huc_col = [c for c in huc8.columns if c.startswith("huc") and c[-1].isdigit()]
    if huc_col:
        huc8 = huc8.drop_duplicates(subset=huc_col[0]).reset_index(drop=True)
    huc8 = gpd.GeoDataFrame(huc8, crs=CRS_GEOGRAPHIC)
    print(f"\nTotal unique HUC-8 watersheds: {len(huc8)}")
else:
    huc8 = gpd.GeoDataFrame()
    print("\nNo HUC-8 data loaded.")

## Load Stream Network

In [ ]:
# Load the stream network for the configured hydrologic context.
# This shared loader uses the local cache before making a public API request.
try:
    streams_gdf = load_nhd_flowlines(STUDY_BBOX, min_stream_order=1)
    print(f"Stream segments in hydrologic context: {len(streams_gdf):,}")
except Exception as error:
    warnings.warn(f"Stream network unavailable: {error}", UserWarning)
    streams_gdf = gpd.GeoDataFrame()


## Public USGS Monitoring Coverage

In [ ]:
# Query USGS groundwater monitoring well inventory
# for the full Pine Ridge hydrologic context

print("Querying USGS groundwater monitoring well inventory...")
gw_sites = load_usgs_groundwater_sites(bbox=STUDY_BBOX)

print(f"USGS groundwater monitoring wells in study area: {len(gw_sites)}")
if not gw_sites.empty:
    print()
    print("Site type breakdown:")
    if "site_tp_cd" in gw_sites.columns:
        print(gw_sites["site_tp_cd"].value_counts().to_string())

In [ ]:
# Compute screening-level monitoring density around the study boundary
coverage_records = []

study_boundary_proj = study_boundary.to_crs(CRS_PROJECTED)
if not gw_sites.empty:
    sites_proj = gw_sites.to_crs(CRS_PROJECTED)

for _, nation in study_boundary_proj.iterrows():
    area_km2 = nation.geometry.area / 1e6
    n_wells  = 0
    nearest_km = None

    if not gw_sites.empty:
        dists    = sites_proj.geometry.distance(nation.geometry.centroid) / 1000
        n_wells  = int((dists <= 50).sum())
        nearest_km = round(float(dists.min()), 1) if len(dists) > 0 else None

    coverage_records.append({
        "common_name":     nation["common_name"],
        "area_km2":        round(area_km2, 0),
        "wells_within_50km": n_wells,
        "nearest_well_km": nearest_km,
        "density_per_1000km2": round(n_wells / (area_km2/1000), 2) if area_km2 > 0 else 0,
        "is_study_area":   nation["common_name"] in STUDY_NAMES,
    })

coverage_df = pd.DataFrame(coverage_records)

print("USGS GROUNDWATER MONITORING DENSITY BY NATION")
print(
    coverage_df[["common_name", "nearest_well_km",
                  "wells_within_50km", "density_per_1000km2"]]
    .sort_values("nearest_well_km", ascending=False)
    .to_string(index=False)
)
print()
print("Nations with nearest USGS well > 50 km = monitoring gap.")
print("This describes the coverage of the selected public dataset; causes and equity implications require additional evidence and review.")

## Watershed-Territory Overlap Analysis

In [ ]:
# Which HUC-8 watersheds overlap each Pine Ridge study boundary?
# This reveals the cross-jurisdictional complexity of water governance.

if not huc8.empty:
    huc_col = [c for c in huc8.columns if c.startswith("huc") and c[-1].isdigit()]
    huc_id  = huc_col[0] if huc_col else None

    study_boundary_proj = study_boundary.to_crs(CRS_PROJECTED)
    huc_proj   = huc8.to_crs(CRS_PROJECTED)

    print("WATERSHED–TERRITORY OVERLAP")
    for _, nation in study_boundary_proj.iterrows():
        overlapping = huc_proj[huc_proj.geometry.intersects(nation.geometry)]
        name_col    = "name" if "name" in overlapping.columns else None
        names       = overlapping[name_col].tolist() if name_col else [
            overlapping[huc_id].tolist() if huc_id else ["unknown"]
        ]
        print(f"\n  {nation['common_name']} ({len(overlapping)} HUC-8 units):")
        for n in names[:5]:
            print(f"    {n}")
        if len(names) > 5:
            print(f"    ... and {len(names)-5} more")
else:
    print("HUC-8 data not available for overlap analysis.")

## Visualizations

In [ ]:
# Export the auditable monitoring-coverage table.
# Static map rendering is intentionally deferred because native geospatial
# rendering varies across Windows environments.
coverage_df.to_csv(OUTPUTS_DIR / "pine_ridge_monitoring_coverage.csv", index=False)
print("Saved outputs/pine_ridge_monitoring_coverage.csv")


In [ ]:
# Report the screening-level monitoring gap in tabular form.
display(coverage_df[["common_name", "wells_within_50km", "nearest_well_km", "density_per_1000km2"]])


## Exports

In [ ]:
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

study_boundary.to_file(
    OUTPUTS_DIR/"pine_ridge_census_boundary.geojson", driver="GeoJSON"
)
print("Exported to outputs/pine_ridge_census_boundary.geojson")

coverage_df.to_csv(
    OUTPUTS_DIR/"pine_ridge_monitoring_coverage.csv", index=False
)
print("Exported to outputs/pine_ridge_monitoring_coverage.csv")

if not streams_gdf.empty:
    streams_gdf.to_file(
        OUTPUTS_DIR/"pine_ridge_context_streams.geojson", driver="GeoJSON"
    )
    print("Exported to outputs/pine_ridge_context_streams.geojson")

In [ ]:
print(generate_citations(
    ["census_aiannh", "nhd_flowlines", "wbd_huc",
     "usgs_nwis_groundwater", "usgs_nwis_streamflow"]
))

## Learner Checkpoint

Choose one mapped boundary or feature. Record what it represents, who produced it, and one claim it cannot support.

## Interpretation Protocol

Before writing a conclusion, separate:

1. **Observation:** what the computed public data show, including unit, period, spatial scope, and missingness.
2. **Interpretation:** a plausible explanation, stated with uncertainty.
3. **Additional evidence:** literature, local monitoring, expertise, or validation needed to evaluate that explanation.
4. **Decision authority:** who is authorized to approve publication, thresholds, or management action.

Do not convert monitoring absence, association, a screening flag, or scenario output into a causal, regulatory, health, policy, or community conclusion.

## Contribution Activity

Improve one map label, boundary caveat, source note, or glossary link. Ask a partner whether the revision prevents a likely misinterpretation. Review the change with a partner and record what became clearer or more defensible.

## Evidence Record and Next Step

Record one regenerated result, its source and scope, one transformation, one limitation, and one question requiring more evidence or local knowledge.

Notebook 02 examines the coverage and record characteristics of configured public groundwater monitoring sites.